In [1]:
import time

In [2]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python.vision import GestureRecognizer, GestureRecognizerOptions, GestureRecognizerResult, RunningMode
from mediapipe.framework.formats import landmark_pb2
from mediapipe import solutions

In [3]:
import cv2
import numpy as np

In [4]:
# Global variable to store the latest detection result
latest_result: GestureRecognizerResult | None = None
is_processing: bool = False

In [5]:
# --- CONFIGURATION ET ÉTATS ---
WAITING_TO_START = "WAITING"
COUNTDOWN = "COUNTDOWN"
CAPTURE_GESTURE = "CAPTURE"
GAME_OVER = "GAME_OVER"

In [6]:
state = WAITING_TO_START
scores = {"Player 1": 0, "Player 2": 0}
countdown_start_time = 0
current_countdown_text = ""

In [7]:
def determine_winner(g1, g2):
    # Logique simple : 0: Pierre, 1: Papier, 2: Ciseau (à adapter selon votre modèle custom)
    rules = {"Rock": "Scissors", "Paper": "Rock", "Scissors": "Paper"}
    if g1 == g2: return "Draw"
    if rules.get(g1) == g2: return "Player 1"
    return "Player 2"

In [8]:
def result_callback(result: GestureRecognizerResult, output_image: mp.Image, timestamp_ms: int):
    _, _ = output_image, timestamp_ms  # Unused in this example
    global latest_result, is_processing
    latest_result = result
    for idx, gesture in enumerate(result.gestures):
        print(f'Hand {idx}: {gesture[0].category_name} ({gesture[0].score:.2f})')
    is_processing = False


In [9]:
def draw_landmarks(image, hand_landmarks_list):
    annotated_image = np.copy(image)

    # Loop through the detected hands to visualize.
    for idx in range(len(hand_landmarks_list)):
        hand_landmarks = hand_landmarks_list[idx]

        # Draw the hand landmarks.
        hand_landmarks_proto = landmark_pb2.NormalizedLandmarkList()
        hand_landmarks_proto.landmark.extend([
            landmark_pb2.NormalizedLandmark(x=landmark.x, y=landmark.y, z=landmark.z) for landmark in hand_landmarks
        ])
        solutions.drawing_utils.draw_landmarks(
            annotated_image,
            hand_landmarks_proto,
            solutions.hands.HAND_CONNECTIONS,
            solutions.drawing_styles.get_default_hand_landmarks_style(),
            solutions.drawing_styles.get_default_hand_connections_style()
        )

    return annotated_image

In [10]:
def draw_game_elements(image, gestures_list):
    global state, countdown_start_time, current_countdown_text, scores
    annotated_image = np.copy(image)
    # image_height, image_width, _ = annotated_image.shape
    #
    # for idx in range(len(gestures_list)):
    #     gesture = gestures_list[idx][0]
    #
    #     # Draw different game elements based on the recognized gesture.
    #     if gesture.category_name == 'Thumb_Up':
    #         cv2.rectangle(annotated_image, (0, 0), (image_width, image_height), (0, 255, 0), -1)  # Red circle for 'Fist'
    #     elif gesture.category_name == 'Thumb_Down':
    #         cv2.rectangle(annotated_image, (0, 0), (image_width, image_height), (255, 0, 0), -1)  # Green square for 'Open_Palm'
    #
    #     # cv2.putText(annotated_image, f'Hand {idx}: {gesture.category_name} ({gesture.score:.2f})',
    #     #             (10, 30 + idx * 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

    current_gestures = []
    if latest_result:
        current_gestures = [g[0].category_name for g in gestures_list]

    if state == WAITING_TO_START:
        cv2.putText(annotated_image, "Faites THUMB_UP pour commencer", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        if current_gestures.count("Thumb_Up") >= 2:
            state = COUNTDOWN
            countdown_start_time = time.time()

    elif state == COUNTDOWN:
        elapsed = time.time() - countdown_start_time
        if elapsed < 1: current_countdown_text = "PIERRE"
        elif elapsed < 2: current_countdown_text = "PAPIER"
        elif elapsed < 3: current_countdown_text = "CISEAU !"
        else:
            state = CAPTURE_GESTURE

        cv2.putText(annotated_image, current_countdown_text, (200, 250), cv2.FONT_HERSHEY_DUPLEX, 3, (255, 255, 255), 5)

    elif state == CAPTURE_GESTURE:
        # On attend d'avoir 2 gestes détectés (pour les 2 mains)
        if len(current_gestures) == 2:
            p1_gesture = current_gestures[0]
            p2_gesture = current_gestures[1]
            winner = determine_winner(p1_gesture, p2_gesture)

            if winner != "Draw":
                scores[winner] += 1

            print(f"P1: {p1_gesture} | P2: {p2_gesture} -> Gagnant: {winner}")
            time.sleep(1.5) # Pause pour voir le résultat
            state = COUNTDOWN # Ou retour direct au compte à rebours

        # Vérification Thumb_Down pour quitter
        if current_gestures.count("Thumb_Down") >= 2:
            state = GAME_OVER

    elif state == GAME_OVER:
        cv2.putText(annotated_image, f"SCORE FINAL: P1 {scores['Player 1']} - {scores['Player 2']} P2",
                    (50, 250), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
        cv2.putText(annotated_image, "Quitter dans 5s...", (50, 300), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.imshow('Game', annotated_image)
        cv2.waitKey(5000)

    # Affichage des scores en permanence
    cv2.putText(annotated_image, f"P1: {scores['Player 1']} | P2: {scores['Player 2']}", (10, 450), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2)
    return annotated_image

In [11]:
def annotate_image(image, detection_result):
    gestures_list = detection_result.gestures
    hand_landmarks_list = detection_result.hand_landmarks

    annotated_image = draw_landmarks(image, hand_landmarks_list)
    annotated_image = draw_game_elements(annotated_image, gestures_list)
    return annotated_image

In [ ]:
model_path = 'models/gesture_recognizer.task'
base_options = python.BaseOptions(model_asset_path=model_path)
options = GestureRecognizerOptions(
    base_options=base_options,
    running_mode=RunningMode.LIVE_STREAM,
    num_hands=2,
    result_callback=result_callback,
)

with GestureRecognizer.create_from_options(options) as recognizer:
    start_time = time.time()
    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("Error: Could not open camera.")
        exit()

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            print("Ignoring empty camera frame.")
            continue

        # Convert the BGR image to RGB.
        rgb_image = cv2.cvtColor(cv2.flip(frame, 1), cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

        if not is_processing and state != COUNTDOWN:
            is_processing = True
            recognizer.recognize_async(mp_image, int((time.time() - start_time) * 1000))

        # If there is a latest result, draw it on the image.
        if latest_result:
            rgb_image = annotate_image(rgb_image, latest_result)

        # Convert back to BGR for OpenCV display.
        display_image = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2BGR)
        cv2.imshow('Gesture Recognizer', display_image)

        if cv2.waitKey(5) & 0xFF == ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()

I0000 00:00:1765984925.534936  494582 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1 Pro
W0000 00:00:1765984925.535287  494582 gesture_recognizer_graph.cc:129] Hand Gesture Recognizer contains CPU only ops. Sets HandGestureRecognizerGraph acceleration to Xnnpack.
I0000 00:00:1765984925.535889  494582 hand_gesture_recognizer_graph.cc:250] Custom gesture classifier is not defined.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1765984925.540706  496281 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1765984925.545443  496281 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1765984925.545825  496281 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedbac

Hand 0: Thumb_Up (0.69)
Hand 0: Thumb_Up (0.86)
Hand 0: Thumb_Up (0.89)
Hand 0: Thumb_Up (0.84)
Hand 0: Thumb_Up (0.69)
Hand 0: Thumb_Up (0.84)
Hand 0: Thumb_Up (0.72)
Hand 0: Thumb_Up (0.75)
Hand 0: Thumb_Up (0.52)
Hand 0: None (0.61)
Hand 0: None (0.61)
Hand 0: Thumb_Up (0.58)
Hand 0: Thumb_Up (0.77)
Hand 0: Thumb_Up (0.58)
Hand 0: Thumb_Up (0.63)
Hand 0: Thumb_Up (0.66)
Hand 0: Thumb_Up (0.65)
Hand 0: Thumb_Up (0.64)
Hand 0: Thumb_Up (0.63)
Hand 0: None (0.52)
Hand 0: Thumb_Up (0.50)
Hand 1: Thumb_Up (0.63)
Hand 0: Thumb_Up (0.58)
Hand 1: Thumb_Up (0.65)
P1: Thumb_Up | P2: Thumb_Up -> Gagnant: Draw
Hand 0: None (0.98)
Hand 0: Thumb_Up (0.67)
Hand 0: Thumb_Up (0.66)
Hand 0: Thumb_Up (0.66)
Hand 0: Thumb_Up (0.67)
Hand 0: Thumb_Up (0.67)
Hand 0: Thumb_Up (0.67)
Hand 0: Thumb_Up (0.67)
Hand 0: Thumb_Up (0.67)
Hand 0: Thumb_Up (0.68)
Hand 0: Thumb_Up (0.68)
Hand 0: Thumb_Up (0.67)
Hand 0: Thumb_Up (0.67)
Hand 0: Thumb_Up (0.67)
Hand 0: Thumb_Up (0.67)
Hand 0: Thumb_Up (0.67)
Hand 0: Thu